<a href="https://colab.research.google.com/github/marcolari06-maker/loan-default-prediction/blob/main/Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Acquiring

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import sklearn

github_csv_url = 'https://raw.githubusercontent.com/marcolari06-maker/loan-default-prediction/refs/heads/main/dataset.csv'

try:
    df_github = pd.read_csv(github_csv_url)
    print("Dataset loaded from github repo loan-default-prediction")
    display(df_github.head())
    print(df_github.shape[0])
except Exception as e:
    print(f"Error during loading from github: {e}")

Dataset loaded from github repo loan-default-prediction


,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


148670


# Preprocessing - da sistemare

In [ ]:
if 'Upfront_charges' in df_github.columns:
    df_github = df_github.drop(columns=['Upfront_charges'])
    print("Column 'Upfront_charges' dropped successfully.")
else:
    print("Column 'Upfront_charges' not found in the DataFrame. Skipping drop operation.")

columns_to_drop = ['Upfront_charges', 'rate_of_interest', 'Interest_rate_spread'
                  , 'property_value', 'LTV', 'ID', 'year']

for col in columns_to_drop:
    if col in df_github.columns:
        df_github = df_github.drop(columns=[col])
        print(f"Column '{col}' dropped successfully.")
    else:
        print(f"Column '{col}' not found in the DataFrame. Skipping.")

print("Current columns in DataFrame after dropping:")
print(df_github.columns)

# Reverting to median imputation for dtir1
median_val = df_github['dtir1'].median()
df_github['dtir1'] = df_github['dtir1'].fillna(median_val)

#Income
median_val = df_github['income'].median()
df_github['income'] = df_github['income'].fillna(median_val)

# Sostituisci i valori nulli nella colonna 'loan_limit' con 'Unknown'
df_github['loan_limit'] = df_github['loan_limit'].fillna('Unknown')

print("Valori nulli nella colonna 'loan_limit' sostituiti con 'Unknown'.")
print("Valori unici e loro frequenze nella colonna 'loan_limit' dopo l'imputazione:")
print(df_github['loan_limit'].value_counts(dropna=False))

# Impute 'term' with its median
median_term = df_github['term'].median()
df_github['term'] = df_github['term'].fillna(median_term)
#print(f"Missing values in 'term' after imputation: {df_github['term'].isnull().sum()}")
#print(f"Median value used for 'term': {median_term}")

# Impute 'loan_purpose' with its mode (most frequent value)
## .mode()[0] is used to handle cases where there might be multiple modes
mode_loan_purpose = df_github['loan_purpose'].mode()[0]
df_github['loan_purpose'] = df_github['loan_purpose'].fillna(mode_loan_purpose)
#print(f"\nMissing values in 'loan_purpose' after imputation: {df_github['loan_purpose'].isnull().sum()}")
#print(f"Mode value used for 'loan_purpose': {mode_loan_purpose}")

##Impute 'Neg_ammortization' with its mode
mode_neg_ammortization = df_github['Neg_ammortization'].mode()[0]
df_github['Neg_ammortization'] = df_github['Neg_ammortization'].fillna(mode_neg_ammortization)
#print(f"\nMissing values in 'Neg_ammortization' after imputation: {df_github['Neg_ammortization'].isnull().sum()}")
#print(f"Mode value used for 'Neg_ammortization': {mode_neg_ammortization}")

##Impute 'approv_in_adv'
mode_approv_in_adv = df_github['approv_in_adv'].mode()[0]
df_github['approv_in_adv'] = df_github['approv_in_adv'].fillna(mode_approv_in_adv)

initial_rows = df_github.shape[0]
print(f"Initial number of rows: {initial_rows}")

# Drop rows where 'age' is null
# As confirmed earlier, these are the same rows where 'submission_of_application' is null
if 'age' in df_github.columns:
    df_github.dropna(subset=['age'], inplace=True)
    print(f"Number of rows after dropping nulls in 'age': {df_github.shape[0]}")
    print(f"Rows dropped: {initial_rows - df_github.shape[0]}")
else:
    print("Column 'age' not found. Skipping operation.")

# Verify if 'submission_of_application' still has nulls (it shouldn't if 'age' and
#   'submission_of_application' nulls are coincident)
if 'submission_of_application' in df_github.columns:
    missing_submission = df_github['submission_of_application'].isnull().sum()
    print(f"Missing values in 'submission_of_application' after operation: {missing_submission}")
else:
    print("Column 'submission_of_application' not found.")

# Check for duplicate rows in the entire DataFrame
duplicate_rows = df_github.duplicated().sum()

if duplicate_rows > 0:
    print(f"Number of duplicate rows found: {duplicate_rows}")
    # Optionally, display the duplicate rows
    # display(df_github[df_github.duplicated(keep=False)]) # keep=False shows all occurrences of duplicates
else:
    print("No duplicate rows found in the DataFrame.")

#Redundancies
columns_to_remove = ['construction_type', 'Security_Type']

for col in columns_to_remove:
    if col in df_github.columns:
        df_github = df_github.drop(columns=[col])
        print(f"Column '{col}' removed successfully.")
    else:
        print(f"Column '{col}' not found in the DataFrame. Skipping.")

print("Current columns in DataFrame after removal:")
print(df_github.columns)

print("Missing values analysis")
missing = df_github.isnull().sum()
missing_pct = (missing / len(df_github) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Values': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

# Mostra solo le colonne che hanno almeno 1 NaN
print(missing_df[missing_df['Missing Values'] > 0])

Column 'Upfront_charges' dropped successfully.
Column 'Upfront_charges' not found in the DataFrame. Skipping.
Column 'rate_of_interest' dropped successfully.
Column 'Interest_rate_spread' dropped successfully.
Column 'property_value' dropped successfully.
Column 'LTV' dropped successfully.
Column 'ID' dropped successfully.
Column 'year' dropped successfully.
Current columns in DataFrame after dropping:
Index(['loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose',
       'Credit_Worthiness', 'open_credit', 'business_or_commercial',
       'loan_amount', 'term', 'Neg_ammortization', 'interest_only',
       'lump_sum_payment', 'construction_type', 'occupancy_type', 'Secured_by',
       'total_units', 'income', 'credit_type', 'Credit_Score',
       'co-applicant_credit_type', 'age', 'submission_of_application',
       'Region', 'Security_Type', 'Status', 'dtir1'],
      dtype='object')
Valori nulli nella colonna 'loan_limit' sostituiti con 'Unknown'.
Valori unici e loro freq

# Classification

## Initial setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    RandomizedSearchCV, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score

df = df_github.copy()
# --- feature / target ---
X = df.drop(columns=["Status"])
y = df["Status"]

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical:", num_features)
print("Categorical:", cat_features)

# --- preprocessore colonna per colonna ---
numeric_transformer = Pipeline(steps=[
   #("imputer", SimpleImputer(strategy="median")),   #IMPUTAZIONE MEDIANA
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
   #("imputer", SimpleImputer(strategy="most_frequent")),  #IMPUTAZIONE MODA
    ("encoder", OrdinalEncoder(handle_unknown="ignore"))
])


preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ]
)

# --- schemi di CV comuni ---
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# helper for eviting copy and paste
def nested_random_search(pipe, param_dist, n_iter=20, scoring="roc_auc"):
    rs = RandomizedSearchCV(
        pipe,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=scoring,
        cv=inner_cv,
        n_jobs=-1,
        random_state=42,
        refit=True
    )
    nested_scores = cross_val_score(
        rs, X, y,
        cv=outer_cv,
        scoring=scoring,
        n_jobs=-1
    )
    return rs, nested_scores



Numerical: ['loan_amount', 'term', 'income', 'Credit_Score', 'dtir1']
Categorical: ['loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose', 'Credit_Worthiness', 'open_credit', 'business_or_commercial', 'Neg_ammortization', 'interest_only', 'lump_sum_payment', 'occupancy_type', 'Secured_by', 'total_units', 'credit_type', 'co-applicant_credit_type', 'age', 'submission_of_application', 'Region']


## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

pipe_lr = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=42
    ))
])

param_dist_lr = {
    "clf__C":            [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__class_weight": [None, "balanced"],
    "clf__penalty":      ["l2"],
}

# --- nested CV con RandomizedSearch ---
rs_lr, nested_scores_lr = nested_random_search(pipe_lr, param_dist_lr, n_iter=20)
print(f"LR nested AUC: {nested_scores_lr.mean():.3f} ± {nested_scores_lr.std():.3f}")

# --- GridSearch di rifinitura su tutto il dataset ---
param_grid_lr = {
    "clf__C":            [0.1, 1, 10],
    "clf__class_weight": [None, "balanced"],
}

gs_lr = GridSearchCV(
    pipe_lr,
    param_grid=param_grid_lr,
    scoring="roc_auc",
    cv=inner_cv,
    n_jobs=-1
)
gs_lr.fit(X, y)
best_lr = gs_lr.best_estimator_
print("LR best params:", gs_lr.best_params_)

LR nested AUC: 0.830 ± 0.002
LR best params: {'clf__C': 1, 'clf__class_weight': 'balanced'}


## Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

pipe_dt = Pipeline([
    ("preprocess", preprocess),
    ("clf", DecisionTreeClassifier(random_state=42))
])

param_dist_dt = {
    "clf__max_depth":        [3, 5, 8, 10, 15, None],
    "clf__min_samples_leaf": [1, 5, 10, 20, 50],
    "clf__criterion":        ["gini", "entropy"],
    "clf__class_weight":     [None, "balanced"],
}

rs_dt, nested_scores_dt = nested_random_search(pipe_dt, param_dist_dt, n_iter=30)
print(f"DT nested AUC: {nested_scores_dt.mean():.3f} ± {nested_scores_dt.std():.3f}")

param_grid_dt = {
    "clf__max_depth":        [3, 5, 8],
    "clf__min_samples_leaf": [5, 10, 20],
    "clf__class_weight":     [None, "balanced"],
}

gs_dt = GridSearchCV(
    pipe_dt,
    param_grid=param_grid_dt,
    scoring="roc_auc",
    cv=inner_cv,
    n_jobs=-1
)
gs_dt.fit(X, y)
best_dt = gs_dt.best_estimator_
print("DT best params:", gs_dt.best_params_)

DT nested AUC: 0.862 ± 0.002
DT best params: {'clf__class_weight': 'balanced', 'clf__max_depth': 8, 'clf__min_samples_leaf': 20}


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

pipe_rf = Pipeline([
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

param_dist_rf = {
    "clf__n_estimators":     [100, 200, 300],
    "clf__max_depth":        [5, 10, 20, None],
    "clf__min_samples_leaf": [1, 5, 10, 20],
    "clf__max_features":     ["sqrt", "log2", 0.3],
    "clf__class_weight":     [None, "balanced"],
}

rs_rf, nested_scores_rf = nested_random_search(pipe_rf, param_dist_rf, n_iter=30)
print(f"RF nested AUC: {nested_scores_rf.mean():.3f} ± {nested_scores_rf.std():.3f}")

# usa i best params del RandomizedSearch come centro del grid
rs_rf.fit(X, y)
best_p_rf = rs_rf.best_params_

param_grid_rf = {
    "clf__n_estimators": [best_p_rf["clf__n_estimators"]],
    "clf__max_depth":    [best_p_rf["clf__max_depth"]],
    "clf__min_samples_leaf": [
        max(1, best_p_rf["clf__min_samples_leaf"] - 5),
        best_p_rf["clf__min_samples_leaf"],
        best_p_rf["clf__min_samples_leaf"] + 5
    ],
    "clf__class_weight": [best_p_rf["clf__class_weight"]],
}

gs_rf = GridSearchCV(
    pipe_rf,
    param_grid=param_grid_rf,
    scoring="roc_auc",
    cv=inner_cv,
    n_jobs=-1
)
gs_rf.fit(X, y)
best_rf = gs_rf.best_estimator_
print("RF best params:", gs_rf.best_params_)

## Adaboost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

pipe_ada = Pipeline([
    ("preprocess", preprocess),
    ("clf", AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        random_state=42
    ))
])

param_dist_ada = {
    "clf__n_estimators": [50, 100, 200],
    "clf__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
}

rs_ada, nested_scores_ada = nested_random_search(
    pipe_ada, param_dist_ada, n_iter=20
)
print(f"AdaBoost nested AUC: {nested_scores_ada.mean():.3f} ± {nested_scores_ada.std():.3f}")


rs_ada.fit(X, y)
best_p_ada = rs_ada.best_params_

param_grid_ada = {
    "clf__learning_rate": [
        best_p_gb["clf__learning_rate"] / 2,
        best_p_gb["clf__learning_rate"],
        best_p_gb["clf__learning_rate"] * 2
    ],
    "clf__max_iter": [best_p_gb["clf__max_iter"]],
    "clf__max_depth": [best_p_gb["clf__max_depth"]],
    "clf__class_weight": [best_p_gb["clf__class_weight"]],
}

gs_ada = GridSearchCV(
    pipe_ada,
    param_grid=param_grid_ada,
    scoring="roc_auc",
    cv=inner_cv,
    n_jobs=-1
)
gs_ada.fit(X, y)
best_ada = gs_ada.best_estimator_
print("GB best params:", gs_ada.best_params_)

## Comparison

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt

summary_nested = pd.DataFrame({
    "model": ["LogReg", "DecTree", "RandomForest", "AdaBoost"],
    "nested_auc_mean": [
        nested_scores_lr.mean(),
        nested_scores_dt.mean(),
        nested_scores_rf.mean(),
        nested_scores_ada.mean()
    ],
    "nested_auc_std": [
        nested_scores_lr.std(),
        nested_scores_dt.std(),
        nested_scores_rf.std(),
        nested_scores_ada.std()
    ],
})

print(summary_nested.sort_values("nested_auc_mean", ascending=False))

models_final = [
    ("LogReg",       best_lr),
    ("DecTree",      best_dt),
    ("RandomForest", best_rf),
    ("AdaBoost",    best_ada),
]

rows = []
for name, clf in models_final:
    # AUC media in CV
    auc_scores = cross_val_score(
        clf, X, y,
        cv=outer_cv,
        scoring="roc_auc",
        n_jobs=-1
    )
    # F1 media sulla classe positiva (default=1)
    f1_scores = cross_val_score(
        clf, X, y,
        cv=outer_cv,
        scoring="f1",   # per binary {0,1} usa F1 su 1
        n_jobs=-1
    )
    # Precision media sulla classe positiva
    precision_scores = cross_val_score(
        clf, X, y,
        cv=outer_cv,
        scoring="precision",
        n_jobs=-1
    )
    # Recall media sulla classe positiva
    recall_scores = cross_val_score(
        clf, X, y,
        cv=outer_cv,
        scoring="recall",
        n_jobs=-1
    )
    rows.append({
        "model":      name,
        "auc_mean":   auc_scores.mean(),
        "auc_std":    auc_scores.std(),
        "f1_mean":    f1_scores.mean(),
        "f1_std":     f1_scores.std(),
        "precision_mean": precision_scores.mean(),
        "precision_std":  precision_scores.std(),
        "recall_mean":    recall_scores.mean(),
        "recall_std":     recall_scores.std(),
    })

df_metrics = pd.DataFrame(rows)
print(df_metrics.sort_values("auc_mean", ascending=False))

plt.figure(figsize=(6, 4))
plt.scatter(df_metrics["auc_mean"], df_metrics["f1_mean"])

for _, r in df_metrics.iterrows():
    plt.text(
        r["auc_mean"] + 0.001,
        r["f1_mean"] + 0.001,
        r["model"]
    )

plt.xlabel("ROC-AUC (media CV)")
plt.ylabel("F1 classe default (media CV)")
plt.title("Confronto modelli nel piano AUC–F1")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

NameError: name 'nested_scores_dt' is not defined